# 00.08 — Q-learning knob GUI

<div><span style="display:inline-block;background:#E45756;color:white;font-weight:800;padding:4px 8px;border-radius:5px;margin:3px">ALPHA</span><span style="display:inline-block;background:#00A6A6;color:white;font-weight:800;padding:4px 8px;border-radius:5px;margin:3px">GAMMA</span><span style="display:inline-block;background:#FF6B9A;color:white;font-weight:800;padding:4px 8px;border-radius:5px;margin:3px">EPSILON</span><span style="display:inline-block;background:#9C6ADE;color:white;font-weight:800;padding:4px 8px;border-radius:5px;margin:3px">EXPLORE</span><span style="display:inline-block;background:#2F80ED;color:white;font-weight:800;padding:4px 8px;border-radius:5px;margin:3px">EXPLOIT</span><span style="display:inline-block;background:#ECA82C;color:white;font-weight:800;padding:4px 8px;border-radius:5px;margin:3px">Q</span></div>

In [1]:
from pathlib import Path
import sys, numpy as np, pandas as pd, matplotlib.pyplot as plt
repo=Path.cwd()
if repo.name=="notebooks": repo=repo.parent
sys.path.insert(0,str(repo))
from lineworld import LineWorld
from visuals import *
legend()
import ipywidgets as widgets
from IPython.display import display, clear_output

Use the sliders. This is where α, γ, and ε should finally become separate physical knobs in your head.

In [2]:
alpha_w=widgets.FloatSlider(value=.2,min=.01,max=1,step=.01,description="α learn")
gamma_w=widgets.FloatSlider(value=.95,min=0,max=1,step=.01,description="γ future")
epsilon_w=widgets.FloatSlider(value=.25,min=0,max=1,step=.01,description="ε explore")
episodes_w=widgets.IntSlider(value=300,min=1,max=1000,description="episodes")
seed_w=widgets.IntText(value=7,description="seed")
button=widgets.Button(description="TRAIN",button_style="success"); out=widgets.Output()
def run(alpha,gamma,epsilon,episodes,seed):
    rng=np.random.default_rng(seed); env=LineWorld(); Q=np.zeros((6,2)); returns=[]; explore_n=0; decisions=0
    for ep in range(episodes):
        obs,_=env.reset(); total=0
        while True:
            decisions+=1
            if rng.random()<epsilon: explore_n+=1; action=int(rng.integers(0,2))
            else: action=int(np.argmax(Q[obs]))
            n,r,term,trunc,_=env.step(action); future=0 if term else np.max(Q[n]); target=r+gamma*future; Q[obs,action]+=alpha*(target-Q[obs,action]); total+=r; obs=n
            if term or trunc: break
        returns.append(total)
    return Q,np.array(returns),explore_n/decisions
def click(_):
    with out:
        clear_output(wait=True); Q,ret,ef=run(alpha_w.value,gamma_w.value,epsilon_w.value,episodes_w.value,seed_w.value); print(f"actual explore fraction: {ef:.3f}"); fig=q_heatmap(Q,f"α={alpha_w.value:.2f} γ={gamma_w.value:.2f} ε={epsilon_w.value:.2f}"); plt.show(); plt.figure(figsize=(9,3)); plt.plot(pd.Series(ret).rolling(20,min_periods=1).mean(),color=COLORS["EPISODE"]); plt.title("learning curve"); plt.xlabel("episode"); plt.show(); policy=np.argmax(Q[:5],axis=1); plt.figure(figsize=(8,3)); plt.bar(range(5),policy,color=COLORS["EXPLOIT"]); plt.yticks([0,1],["LEFT","RIGHT"]); plt.title("greedy policy after training"); plt.show()
button.on_click(click); display(widgets.VBox([alpha_w,gamma_w,epsilon_w,episodes_w,seed_w,button,out])); click(None)

Try: α=.01 vs 1.0; γ=0 vs .95 vs 1; ε=0 vs .25 vs 1.0. **Q is what learns; α is update size; γ is future weighting; ε is data-collection randomness.**